# Umsatzprognose

Dieses Notebook zeigt, wie viel Umsatz aus den Projekten zu erwarten ist, die bereits
in Clockodo angelegt sind. Es liest die Daten nur; es verändert in Clockodo nichts.

**So wird es benutzt:** oben im Menü *Laufzeit → Alle ausführen*, dann von oben nach
unten lesen. Der Abruf dauert etwa eine halbe Minute. Alle Diagramme sind interaktiv –
mit dem Mauszeiger über einem Balken stehen die genauen Zahlen.

In [ ]:
import importlib

if (
    importlib.util.find_spec("google") is not None
    and importlib.util.find_spec("google.colab") is not None
):
    PAKET_URL = "git+https://github.com/it-agile/umsatzprognose-clockodo.git"

    !pip install --quiet "$PAKET_URL"
    !pip install --quiet --force-reinstall --no-deps "$PAKET_URL"
else:
    print("Lokale Installation")

## Parameter und Variablen für die Verarbeitung

In [ ]:
from datetime import date

stichtag = date.today()
abgeschlossene_monate = 12
horizont_monate = 3
projekte_ohne_auftragsvolumen = 20

## Daten abrufen

Die Zugangsdaten kommen in Colab aus der Secrets-Verwaltung (Schlüssel-Symbol in der
linken Seitenleiste), lokal aus der Datei `.env`.

In [ ]:
from umsatzprognose import Dashboard

dashboard = Dashboard.laden(
    stichtag=stichtag, abgeschlossene_monate=abgeschlossene_monate, horizont_monate=horizont_monate
)

print(
    f"Abrechnungsdaten geladen.\n"
    f"Stand der Auswertung: {dashboard.stichtag:%d.%m.%Y}\n"
    f"{dashboard.anzahl_schulungen} Schulung(en) geladen\n"
    f"{dashboard.anzahl_kostenmonate} Monat(e) mit Kostenprognose geladen"
)

## Stundensatz für Projekte ohne Umsatz hinterlegen

Manche Projekte oben haben gebuchte Zeit, aber keinen Umsatz - ihr Stundensatz ist 0.
Ohne eine Korrektur bricht die spätere Umrechnung von Euro in Stunden an dieser Stelle.
Betroffene Projekte stehen im Hinweis "Stundensatz 0" oben, mit Namen.

Für jedes betroffene Projekt lässt sich hier ein plausibler Stundensatz nachtragen -
Projektname wie in der Hinweistabelle, Satz in Euro je Stunde. Die Zeile bleibt ohne
Wirkung, solange kein Projekt betroffen ist.

In [ ]:
# Projektname wie in der Hinweistabelle oben, Satz in Euro je Stunde. Beispiel:
# stundensatz_korrekturen = {"Beispielprojekt": 95.0}
stundensatz_korrekturen = {}

dashboard.stundensatz_uebersteuern(stundensatz_korrekturen)

Zur Kontrolle: der Hinweis "Stundensatz 0" verschwindet für die eingetragenen Projekte.

In [ ]:
dashboard.simuliere(monate=horizont_monate)

## Überblick

Links der tatsächlich erzielte Umsatz der vergangenen zwölf abgeschlossenen Monate,
rechts das Volumen, das aus laufenden Projekten noch abgerufen werden kann.

In [ ]:
dashboard.kennzahlen()

## Umsatz, Kosten und Gewinn je Monat

Alle Buchungen des jeweiligen Monats, auch die ohne Projektbezug. Der letzte Balken
der Historie ist der laufende Monat.

Rechts daran schließt sich der Prognosehorizont an: 
* **bereits gebuchter** Umsatz je Monat
* **prognostizierter** Umsatz obendrauf in gedämpfter Farbe
* **Schulungsanmeldungen**, sofern unten geladen: der schon feststehende Umsatz
  bereits geplanter öffentlicher Schulungstermine, additiv und ohne eigene Bandbreite.

Der dünne Balken am oberen Rand zeigt, wie weit die vorsichtigeren 85-%- und 95-%-Schätzungen darunter liegen.

Zusätzlich, sofern eine Kostenprognose geladen ist, läuft eine **Kosten**-Linie über
die gesamte Breite - Historie und Prognosehorizont gleichermaßen, denn anders als der
Umsatz gilt die Kostenprognose auch für bereits vergangene Monate. Auch sie hat keine
eigene Bandbreite: der Wert steht in der externen Kostenplanung schon fest. Die Lücke
zwischen Umsatzbalken und Kostenlinie ist der **Gewinn**; in der Tabelle darunter steht
er als eigene Spalte, zusammen mit den Kosten.

Der Umsatz bereits geplanter öffentlicher Schulungstermine sowie die Kostenprognose kommen aus derselben separaten Google-Sheets-Tabelle (unterschiedliche Tabellenblätter je Baustein) - ein Login deckt beides ab. In Colab meldest du dich dafür mit deinem eigenen Google-Konto an (ein Login-Fenster öffnet sich); lokal öffnet der erste Aufruf einmalig einen Browser-Tab zum Anmelden, danach läuft es automatisch. Ohne eingetragene Zugangsdaten (siehe `.env.sample` bzw. die passenden Colab-Secrets) bricht die folgende Zelle mit einer Fehlermeldung ab.

In [ ]:
dashboard.umsatzverlauf()

Dieselben Zahlen zum Nachlesen:

In [ ]:
dashboard.umsatztabelle()

## Offenes Auftragsvolumen

Je Projekt der Teil des beauftragten Volumens, der noch nicht verbraucht ist.

In [ ]:
dashboard.restvolumen_je_projekt(top=projekte_ohne_auftragsvolumen)

## Was zu den Zahlen zu wissen ist

Nicht jedes Projekt lässt sich auswerten. Hier steht, welche Fälle aufgetreten sind
und was sie für die Zahlen oben bedeuten.

In [ ]:
dashboard.hinweise()

## Projekte ohne Budget
 Ohne gefilterte Projekte, sortiert.

In [ ]:
# filter kann hier direkt angepasst werden
filter = [
    "it-agile GmbH",
    "Öffentliche Schulung",
]

dashboard.projekte_ohne_budget(filter=filter)